In [1]:
"""
Alpha Trading Research Platform

Notebook:
Sprint09_Paper_Trading

Purpose:
The lightweight paper-trading workflow: scan for opportunities, size
a position against the 1% risk rule, log it to the journal. Run this
top to bottom each time you want to open a paper trade; use the
closing cell separately whenever you're closing one out.

Author:
Alison

Version:
0.11
"""

"\nAlpha Trading Research Platform\n\nNotebook:\nSprint09_Paper_Trading\n\nPurpose:\nThe lightweight paper-trading workflow: scan for opportunities, size\na position against the 1% risk rule, log it to the journal. Run this\ntop to bottom each time you want to open a paper trade; use the\nclosing cell separately whenever you're closing one out.\n\nAuthor:\nAlison\n\nVersion:\n0.11\n"

In [2]:
from alpha.config import Config
from alpha.universes import load_universe
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.regime import calculate_regime
from alpha.scanner import scan_latest, top_opportunities
from alpha.position_sizing import calculate_position_size, calculate_position_size_rebalance_only
from alpha.trade_journal import (
    log_trade_open,
    log_trade_close,
    get_open_trades,
    get_closed_trades,
    summarize_journal,
    load_journal,
)

In [3]:
# =====================================================
# LOAD THE UNIVERSE FROM FILE
# =====================================================
# universes/ftse100.csv is a STARTING POINT, not a verified live list -
# see the comment header in that file. Cross-check it against an
# authoritative source (e.g. londonstockexchange.com) before trusting
# it, and edit the CSV directly once you have a verified list - no
# code changes needed here.

ftse_tickers = load_universe("../universes/ftse100.csv")
print(f"Loaded {len(ftse_tickers)} tickers")
print(ftse_tickers[:5], "...")

Loaded 89 tickers
['III.L', 'ADM.L', 'AAL.L', 'ANTO.L', 'AHT.L'] ...


In [4]:
config = Config(
    universe=ftse_tickers,
    top_stocks=10,
    regime_benchmark="^FTSE",
)
config

# Set this to your actual paper trading account size - NOT stored in
# Config since it's personal and changes over time, not something
# worth committing to git alongside the code.
ACCOUNT_SIZE = 10000

# One file, plain CSV, sitting next to this notebook's parent folder.
# Not committed to git by default - see the .gitignore note in the
# README if you want to change that.
JOURNAL_PATH = "../paper_trading/journal.csv"

print(f"Account size: {ACCOUNT_SIZE}, risk per trade: {config.risk_per_trade:.1%}")

Account size: 10000, risk per trade: 1.0%


In [5]:
# =====================================================
# SCAN
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

scan = scan_latest(monthly_prices, regime, config)
shortlist = top_opportunities(scan, n=3)
display(shortlist)

[**********************48%                       ]  43 of 89 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PHNX.L"}}}
$PHNX.L: possibly delisted; no timezone found
[**********************91%*******************    ]  81 of 89 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHT.L"}}}
[**********************96%*********************  ]  85 of 89 completed$AHT.L: possibly delisted; no timezone found
[*********************100%***********************]  89 of 89 completed

2 Failed downloads:
['PHNX.L', 'AHT.L']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed


,as_of,ticker,direction,score,strategies,conflicting_signal
0,2026-07-31,HSBA.L,long,3,"Momentum, Trend Following, Breakout",False
1,2026-07-31,LLOY.L,long,3,"Momentum, Trend Following, Breakout",False
2,2026-07-31,STAN.L,long,3,"Momentum, Trend Following, Breakout",False


In [6]:
# =====================================================
# SIZE A POSITION
# =====================================================
# Pick one ticker from the shortlist above and fill these in by hand.
# stop_loss is YOUR decision - this package doesn't set one for you.
# A common starting point is a fixed % below entry (e.g. 5-10% for a
# long) or below a recent swing low, but that's a judgement call, not
# something calculated here.

TICKER = "HBSA.L"          # <- change this
DIRECTION = "long"       # <- "long" or "short"
STRATEGY = "Momentum"    # <- which strategy flagged it
ENTRY_PRICE = 15.05     # <- current/planned entry price
ASSUMED_ADVERSE_MOVE_PCT = 0.08  # <- risk buffer, not an exit order

sizing = calculate_position_size_rebalance_only(
    account_size=ACCOUNT_SIZE,
    entry_price=ENTRY_PRICE,
    assumed_adverse_move_pct=ASSUMED_ADVERSE_MOVE_PCT,
    direction=DIRECTION,
    config=config,
)
STOP_LOSS = None  # no real stop-loss order for a rebalance-only trade

# --- Alternative: stop-loss based sizing instead --------------------
# STOP_LOSS = 180.00
# sizing = calculate_position_size(
#     account_size=ACCOUNT_SIZE,
#     entry_price=ENTRY_PRICE,
#     stop_loss=STOP_LOSS,
#     direction=DIRECTION,
#     config=config,
# )

print(f"Shares to buy/short: {sizing.shares}")
print(f"Position value: {sizing.position_value:,.2f}")
print(f"% of account: {sizing.position_pct_of_account:.1%}")
print(f"Risk amount: {sizing.risk_amount:,.2f}")
if sizing.capped_by_max_position:
    print(f"Note: capped by max_position_pct ({config.max_position_pct:.0%}) - "
          f"the risk calculation alone would have sized this larger.")

Shares to buy/short: 83
Position value: 1,249.15
% of account: 12.5%
Risk amount: 99.93


In [10]:
# =====================================================
# LOG THE TRADE
# =====================================================
# Only run this cell once you've actually decided to take the trade -
# it writes a new row to the journal every time it runs.

trade_id = log_trade_open(
    JOURNAL_PATH,
    ticker=TICKER,
    direction=DIRECTION,
    strategy=STRATEGY,
    entry_price=ENTRY_PRICE,
    shares=sizing.shares,
    account_size=ACCOUNT_SIZE,
    risk_amount=sizing.risk_amount,
    stop_loss=STOP_LOSS,  # None for a rebalance-only trade
    notes="",  # add anything worth remembering about why you took this
)

print(f"Logged. trade_id = {trade_id}")
print("Keep this trade_id somewhere - you'll need it to close the trade later.")

Logged. trade_id = HBSA.L-20260722194753
Keep this trade_id somewhere - you'll need it to close the trade later.


In [ ]:
# =====================================================
# CURRENT JOURNAL STATE
# =====================================================

print("Open trades:")
display(get_open_trades(JOURNAL_PATH))

print("\nClosed trades:")
display(get_closed_trades(JOURNAL_PATH))

print("\nSummary:")
print(summarize_journal(JOURNAL_PATH))

In [ ]:
# =====================================================
# CLOSING A TRADE (run this separately, whenever you're ready)
# =====================================================
# Paste in the trade_id you saved earlier, and the price you're
# closing at.

# trade_id_to_close = "AAPL-20260719120000"  # <- uncomment and fill in
# exit_price = 205.00                          # <- uncomment and fill in
# log_trade_close(JOURNAL_PATH, trade_id_to_close, exit_price)
# print("Closed.")

In [ ]:
# =====================================================
# NOTES
# =====================================================
# This is deliberately lightweight - not the full Sprint 10 (Portfolio
# Management). What's NOT covered here: sector exposure, cash
# allocation across multiple simultaneous positions, correlation
# between holdings, or automated stop-loss placement. If you open
# several positions at once, nothing here checks whether they're all
# in the same sector or otherwise correlated - that's still on you to
# judge, or a future sprint to automate.
#
# max_position_pct (20% default) is a blunt safety cap, not a
# diversification strategy - it stops any single position from being
# oversized, it doesn't manage the portfolio as a whole.